# 🎙️ F5-TTS Türkçe — 10 Saniyede Sıfır Örnekli (Zero-Shot) Ses Klonlama

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beydah/ses-klonlama-turkce-f5-tts/blob/main/Ses-Klonlama_Turkce_F5-TTS.ipynb)
[![Website](https://img.shields.io/badge/Website-beydahsaglam.com-blue?style=flat&logo=googlechrome)](https://beydahsaglam.com)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-beydah-0A66C2?style=flat&logo=linkedin)](https://linkedin.com/in/beydah)
[![GitHub](https://img.shields.io/badge/GitHub-beydah-181717?style=flat&logo=github)](https://github.com/beydah)

Bu Colab çalışma alanı, **F5-TTS (Flow Matching)** mimarisi ve topluluk tarafından eğitilen Türkçe ağırlıklar (`marduk-ra/F5-TTS-Turkish`) ile **yalnızca 10 saniyelik tek bir ses kaydınızdan** kendi sesinizi birebir klonlamanızı sağlar.

### 🚀 Neden F5-TTS ve Akış Eşleme (Flow Matching)?
- **Eğitim Gerektirmez (Zero-Shot):** Saatlerce stüdyo kaydına, günlerce GPU kiralamaya veya LoRA/SFT eğitimine ihtiyaç duymaz.
- **Doğal ve Akıcı:** Eski nesil robotik seslerin veya difüzyondaki fısıltıların aksine, ses tellerinizin titreşimini ve konuşma temponuzu anında yakalar.
- **5 Basit Adım:** Sesi hazırla, yükle, temizle, metni ver ve klonlanmış stüdyo kaydını indir!

In [ ]:
#@title ⚙️ 0. Gerekli Kütüphanelerin Kurulumu (Tek Tıkla Çalıştırın)
#@markdown F5-TTS, Vocos ve ses işleme araçları Colab ortamına kurulur (~40 saniye).

import subprocess

print("⏳ Gerekli paketler yükleniyor (f5-tts, pydub, soundfile, scipy)...")
subprocess.run(["pip", "install", "-q", "f5-tts", "pydub", "soundfile", "scipy"], check=True)

# Colab torchvision / transformers uyumsuzluk koruması
import transformers
transformers.pipeline = lambda *args, **kwargs: None

print("✅ Kurulum tamamlandı! Şimdi 1. Adıma geçebilirsiniz.")

---
## 🎯 1. Adım: Referans Sesinizi Hazırlayın

Model, ses tonunuzu ve diksiyonunuzu **10 saniyelik** tek bir referans ses kaydından kopyalayacaktır.

### 🎙️ Okunacak 29 Harfli Altın Cümle:
> **"Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir."**

*(Bu cümle Türk alfabesindeki 29 harfin tamamını barındırır; model sesinizin tüm harf tınılarını tek seferde öğrenir.)*

### 💡 Kaliteli Ses İpuçları:
1. **Süre:** Yaklaşık 8 ile 12 saniye arasında okuyun.
2. **Mikrofon:** Mikrofonu ağzınızın tam karşısına değil, hafifçe yanak hizanıza (15-20 cm mesafede) tutun.
3. **Ton:** Fısıldamayın veya yapay bir ciddiyet takınmayın; sanki bir dostunuza heyecanla bir fikir anlatıyormuş gibi enerjik, net ve tok konuşun.

In [ ]:
#@title 📤 2. Adım: Referans Sesinizi Yükleyin
#@markdown Çalıştırın ve açılan butondan kaydettiğiniz ses dosyasını seçin (.wav, .mp3, .ogg, .m4a):

import os
from google.colab import files

print("🎙️ Lütfen 8-12 saniyelik referans sesinizi seçin:")
yuklenen = files.upload()

if not yuklenen:
    raise ValueError("⚠️ Lütfen bir ses dosyası yükleyin!")

ham_ses_yolu = list(yuklenen.keys())[0]
print(f"✅ Ses dosyası yüklendi: {ham_ses_yolu}")

In [ ]:
#@title 🎛️ 3. Adım: Sesi Otomatik Temizleyin ve İyileştirin
#@markdown Bu aşama; baş ve sondaki sessizlikleri keser, 75 Hz altı dip uğultularını temizler, çıtırtıları yok eder ve stüdyo seviyesine (-1.0 dBFS) normalize eder.

import soundfile as sf
import numpy as np
from scipy import signal
from pydub import AudioSegment
from IPython.display import Audio, display

# Pydub ile tüm ses formatlarını otomatik olarak 48kHz mono WAV'a çevir
seg = AudioSegment.from_file(ham_ses_yolu)
seg = seg.set_channels(1).set_frame_rate(48000)
temp_wav = "temp_raw.wav"
seg.export(temp_wav, format="wav")

# Soundfile ile oku
data, sr = sf.read(temp_wav)
if len(data.shape) > 1:
    data = data.mean(axis=1)

# 1. 75 Hz High-pass Butterworth Filtresi (Dip uğultusu ve nefes patlamasını süzer)
sos = signal.butter(4, 75, 'hp', fs=sr, output='sos')
filtered = signal.sosfilt(sos, data)

# 2. Akıllı Sessizlik Tespiti ve Kırpma (VAD)
frame_len = int(sr * 0.03)  # 30ms pencereler
energy = np.array([np.sum(filtered[i:i+frame_len]**2) for i in range(0, len(filtered)-frame_len, frame_len)])
thresh = np.percentile(energy, 65) * 0.08  # dinamik enerji eşiği
active_frames = np.where(energy > thresh)[0]

if len(active_frames) > 0:
    start_idx = max(0, active_frames[0] * frame_len - int(sr * 0.08))
    end_idx = min(len(filtered), (active_frames[-1] + 1) * frame_len + int(sr * 0.12))
    trimmed = filtered[start_idx:end_idx]
else:
    trimmed = filtered

# 3. Yumuşak Kenar Geçişleri (Fade in/out ile mikro-tıklamaları engeller)
fade_in = int(sr * 0.015)
fade_out = int(sr * 0.025)
if len(trimmed) > (fade_in + fade_out):
    trimmed[:fade_in] *= np.linspace(0, 1, fade_in)
    trimmed[-fade_out:] *= np.linspace(1, 0, fade_out)

# 4. Normalizasyon (-1.0 dBFS tepe noktası)
peak = np.max(np.abs(trimmed))
if peak > 0:
    trimmed = trimmed / peak * 0.89

temiz_ses_yolu = "temiz_referans.wav"
sf.write(temiz_ses_yolu, trimmed, sr)
print(f"✨ Ses başarıyla iyileştirildi! Süre: {len(trimmed)/sr:.2f} saniye")
print("🎧 Temizlenen referans sesi dinleyin:")
display(Audio(temiz_ses_yolu, rate=sr))

---
## 🧠 4. Adım: Metni Girin ve Sesi Klonlayın

Aşağıdaki forma seslendirmek istediğiniz **hedef metni** yapıştırın.

* **Sıfır Halüsinasyon Mimarisi:** Model, metni bağımsız tam cümleler halinde (`.`, `!`, `?`) işler. Her cümlenin arasına **250 milisaniyelik doğal insan nefesi** yerleştirir.
* Bu yöntem; uzun metinlerde ortaya çıkan kelime uydurma, harf yutma veya fısıldama gibi kusurları tamamen ortadan kaldırır.

In [ ]:
#@title 🚀 4. Adım: F5-TTS ile Sesi Klonlayın
#@markdown Seslendirmek istediğiniz metni ve konuşma ayarlarını girin:

import time
import re
import os
import torch
import soundfile as sf
import numpy as np
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process

# 1. Metin ve Referans Bilgileri
#@markdown **Referans Okunan Cümle (Varsayılan Altın Cümle):**
ref_text = "Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir." #@param {type:"string"}

#@markdown **Klonlanacak Hedef Metin:**
hedef_metin = """
Bir modelin cevabını ekranda göstermek ile modelin arayüz kodunu çalıştırmasına izin vermek farklı kararlardır.
Kişisel bir uygulamada ilk deneme için küçük, önceden tanımlanmış bileşenler ve doğrulanmış veri daha denetlenebilir bir başlangıç sunar.
Bu yazı üretimde ölçülmüş bir performans sonucu değil, bir uygulama tasarımıdır.
""" #@param {type:"string"}

#@markdown **Gelişmiş Ayarlar:**
cfg_strength = 2.2 #@param {type:"slider", min:1.0, max:3.0, step:0.1}
nfe_step = 32 #@param {type:"slider", min:16, max:64, step:4}

# 2. Modelleri Yükle (İlk çalıştırmada otomatik indirilir)
if 'f5_model' not in globals():
    print("⏳ F5-TTS Türkçe modeli belleğe yükleniyor...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    from huggingface_hub import hf_hub_download
    ckpt_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="f5_tts_turkish_1000000.safetensors")
    vocab_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="vocab.txt")
    
    model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
    f5_model = load_model(DiT, model_cfg, ckpt_path, mel_spec_type="vocos", vocab_file=vocab_path, device=device)
    f5_vocoder = load_vocoder(vocoder_name="vocos", is_custom=False, device=device)
    print(f"✅ Model hazır! Çalışma ortamı: {device.upper()}")

# 3. Metni Cümle Cümle İşle (Doğal insan nefesi & Kesintisiz Diksiyon)
cumleler = [c.strip() for c in re.split(r'(?<=[.!?])\\s+', hedef_metin.replace('\\n', ' ')) if c.strip()]
print(f"📖 Toplam {len(cumleler)} cümle seslendirilecek...")

ses_parcalari = []
sr_val = 24000
nefes_duraklamasi = np.zeros(int(0.25 * sr_val)) # Cümleler arası 250ms doğal nefes payı

baslangic = time.time()
for idx, cumle in enumerate(cumleler, 1):
    print(f"[{idx}/{len(cumleler)}] Seslendiriliyor: \"{cumle[:50]}...\"")
    audio_segment, final_sample_rate, _ = infer_process(
        temiz_ses_yolu,
        ref_text,
        cumle,
        f5_model,
        f5_vocoder,
        mel_spec_type="vocos",
        nfe_step=nfe_step,
        cfg_strength=cfg_strength,
        speed=1.0,
        device=device
    )
    sr_val = final_sample_rate
    ses_parcalari.append(audio_segment)
    ses_parcalari.append(nefes_duraklamasi)

tam_ses = np.concatenate(ses_parcalari)
cikti_dosyasi = "turkce_klonlanmis_ses.wav"
sf.write(cikti_dosyasi, tam_ses, sr_val)

toplam_sure = len(tam_ses) / sr_val
gecen_zaman = time.time() - baslangic
print(f"🎉 Ses üretimi tamamlandı! Toplam Ses Süresi: {toplam_sure:.1f} sn | Üretim Süresi: {gecen_zaman:.1f} sn")

In [ ]:
#@title 📥 5. Adım: Sesi Dinle ve Bilgisayarına İndir
#@markdown Üretilen sesi dinleyin ve otomatik olarak bilgisayarınıza indirin:

from IPython.display import Audio, display
from google.colab import files

display(Audio(cikti_dosyasi, rate=sr_val))
files.download(cikti_dosyasi)
print("✅ Dosya indirme işlemi başlatıldı!")

---
### 🔗 Bağlantılar ve İletişim

Bu proje, açık kaynaklı **F5-TTS** mimarisinin Türkçe ağırlıkları (`marduk-ra/F5-TTS-Turkish`) üzerinde sıfır örnekli (zero-shot) ses klonlama deneyimini herkes için kolaylaştırmak amacıyla geliştirilmiştir.

* 🌐 **Web Sitesi & Blog:** [beydahsaglam.com](https://beydahsaglam.com)
* 💼 **LinkedIn:** [linkedin.com/in/beydah](https://linkedin.com/in/beydah)
* 🐙 **GitHub:** [github.com/beydah](https://github.com/beydah)
* 📄 **F5-TTS Makalesi:** [F5-TTS: A Fairytaler that Fakes Fluent and Faithful Speech with Flow Matching](https://arxiv.org/abs/2410.06885)